# Experiment 8: Hopfield Network (Store 4 Vectors)

## Aim
To design a Hopfield Network to store and recall 4 patterns.

## Objective
To understand auto-associative memory using Hopfield networks.

## Theory

### What is a Hopfield Network?

A **Hopfield Network** is a type of recurrent neural network that:
- Stores patterns as **memories** using weights
- **Recalls** stored patterns from noisy or partial input
- Finds the closest matching stored pattern (like pattern completion)
- Uses **auto-associative memory** (input and output are same space)

### Key Characteristics

**Recurrent:** Output feeds back as input (unlike feedforward networks)

**Symmetric Weights:** W[i][j] = W[j][i]
- Weight from neuron i to j equals weight from j to i
- Creates stable energy landscape for convergence

**No Self-Connections:** W[i][i] = 0
- Neuron doesn't have weight to itself
- Prevents permanent activation

**Bipolar Values:** Neurons use -1 and +1 (not 0 and 1)
- Simplifies mathematical analysis
- Symmetric around zero

### Weight Formula (Hebbian Rule)

$$W = \sum_{p=1}^{n} x_p \cdot x_p^T$$

Where:
- Each pattern x_p contributes to the weight matrix
- Outer product stores pattern associations
- W is accumulated sum of all pattern contributions

**Intuition:** "Neurons that fire together, wire together"

### Recall Process (Convergence)

1. Provide noisy/partial pattern as input
2. Update neurons: x_new = sign(W · x_old)
3. Repeat until pattern stabilizes (converges)
4. Converged pattern is the recalled memory closest to input


In [1]:

import numpy as np


## Step 1: Define Storage Patterns

We create 4 distinct patterns to store in the Hopfield network.

**Why 4 patterns?**
- Sufficient to demonstrate multiple memories
- Shows that network can distinguish between different patterns
- Computationally manageable (4×4 weight matrix)

**Pattern Format (Bipolar):**
- Uses -1 and +1 instead of 0 and 1
- Each pattern is a 4-dimensional vector
- Example: [1, -1, 1, -1] means (active, inactive, active, inactive)

**These patterns will become the network's "memories"** that it can recall even when given corrupted versions.


In [2]:

# Define 4 patterns (bipolar: -1, 1)
patterns = np.array([
    [1, -1, 1, -1],
    [-1, 1, -1, 1],
    [1, 1, -1, -1],
    [-1, -1, 1, 1]
])

print("Stored Patterns:")
print(patterns)


Stored Patterns:
[[ 1 -1  1 -1]
 [-1  1 -1  1]
 [ 1  1 -1 -1]
 [-1 -1  1  1]]


In [3]:

# Initialize weight matrix
n = patterns.shape[1]
W = np.zeros((n, n))

# Hebbian learning
for p in patterns:
    W += np.outer(p, p)

# Remove self connections
np.fill_diagonal(W, 0)

print("Weight Matrix:")
print(W)


Weight Matrix:
[[ 0.  0.  0. -4.]
 [ 0.  0. -4.  0.]
 [ 0. -4.  0.  0.]
 [-4.  0.  0.  0.]]


## Step 2: Compute Weight Matrix (Hebbian Learning)

The weight matrix stores all 4 patterns using **Hebbian learning rule**.

**Process:**

1. **Outer Product:** For each pattern, compute x · x^T
   - Creates a matrix encoding pattern associations
   - Example: pattern [1, -1] gives outer product of how neurons should connect

2. **Accumulation:** W = W1 + W2 + W3 + W4
   - Sum all outer products
   - Each pattern contributes to the final weights
   - Creates interference but still retrieves correct patterns

3. **Remove Self-Connections:** Set diagonal to 0
   - Prevents neurons from remembering themselves
   - Forces retrieval through network dynamics

**Why This Works:**
- Stored patterns correspond to stable energy minima
- Noisy input naturally slides toward nearest stable state
- Like water flowing downhill to valleys (stored patterns)


In [4]:

def sign(x):
    return np.where(x >= 0, 1, -1)


## Step 3: Define Activation Function

The **sign function** converts continuous values to bipolar outputs.

**Function:**
- If x ≥ 0: output = +1 (active/on)
- If x < 0: output = -1 (inactive/off)

**Purpose:**
- Neurons in Hopfield must be bipolar (-1 or +1)
- The sign function rounds continuous values to nearest pole
- Enables pattern storage and retrieval

**Example:**
- sign(0.5) = +1
- sign(-0.3) = -1
- sign(0) = +1


In [5]:

# Test with noisy input
test = np.array([1, -1, 1, 1])  # slightly corrupted

print("Initial Input:", test)

for i in range(5):
    test = sign(np.dot(W, test))
    print(f"Iteration {i+1}:", test)


Initial Input: [ 1 -1  1  1]
Iteration 1: [-1 -1  1 -1]
Iteration 2: [ 1 -1  1  1]
Iteration 3: [-1 -1  1 -1]
Iteration 4: [ 1 -1  1  1]
Iteration 5: [-1 -1  1 -1]


## Step 4: Pattern Recall with Noisy Input

Now test if the network can recall stored patterns from corrupted input.

**Test Scenario:**
- Provide: [1, -1, 1, 1] (corrupted version)
- Stored pattern 1: [1, -1, 1, -1] (last bit differs!)
- Network should converge back to pattern 1

**Convergence Process:**

1. **Iteration 1:** Update each neuron using x_new = sign(W · x_old)
   - Each neuron receives weighted sum of current state
   - Applies sign function to round to -1 or +1

2. **Iterations 2-5:** Pattern gradually converges
   - Updates move pattern closer to stored memory
   - Network "relaxes" into stable state
   - Like dropping a ball that rolls to nearest valley

**Expected Behavior:**
- Last bit flips from +1 to -1
- Pattern converges to stored pattern [1, -1, 1, -1]
- Convergence typically happens within a few iterations

**Key Insight:** 
- This is **pattern completion** - corrupted input becomes clear
- Like recognizing a face even with half hidden
- Useful for error correction and memory systems


## Summary & Key Takeaways

### What We Accomplished

1. **Pattern Storage:** Stored 4 patterns using Hebbian learning rule
2. **Weight Matrix Creation:** Computed symmetric weight matrix encoding all patterns
3. **Pattern Recall:** Successfully retrieved stored patterns from noisy input
4. **Convergence:** Demonstrated network's ability to converge to stable states

### Key Concepts

| Term | Definition |
|------|-----------|
| **Auto-Associative** | Input and output are from same space; network recalls patterns given noisy versions |
| **Hebbian Learning** | "Neurons that fire together, wire together"; weight = outer product of patterns |
| **Recurrent** | Output feeds back as input; enables iterative convergence |
| **Convergence** | Network reaches stable pattern after several iterations |
| **Energy Landscape** | Mathematical model where stored patterns are valleys; network slides downhill to closest valley |

### How Hopfield Networks Work

```
Store Phase:
    Patterns → Outer Products → Sum Weights → Remove Diagonal
                    ↓
Recall Phase:
    Noisy Input
        ↓
    Multiply by W: x_new = sign(W · x)
        ↓
    Iterate until convergence
        ↓
    Clean Pattern Retrieved
```

### Why This Matters

**Hopfield networks demonstrate:**
- **Pattern completion** - reconstructing full patterns from partial info
- **Error correction** - recovering original signals from noise
- **Content-addressable memory** - recall by partial match, not address
- **Biological plausibility** - inspired by actual neural networks

### Real-World Applications

1. **Image Restoration:** Remove noise from corrupted images or photos
2. **Error Correction:** Recover data corrupted during transmission
3. **Pattern Recognition:** Identify objects despite occlusion or distortion
4. **Associative Retrieval:** "Like finding a file when you partially remember it"

### Limitations

- Limited storage capacity (can store roughly 0.14 × n patterns in n neurons)
- With many patterns, recall may fail or converge to spurious patterns
- Cannot distinguish patterns that are too similar

### Conclusion

We successfully implemented a Hopfield Network and demonstrated how it stores and recalls patterns using Hebbian learning and symmetric weights. This foundational architecture showcases the power of **associative memory** and **recurrent processing** for robust pattern recall from corrupted or incomplete input.
